LAYER 2 : REASONING ENGINE
This is where the agent starts reasoning like a human.
The notebook covers Decision intelligence across:

compatibility reasoning, contextual reasoning, rating prediction, review planning, recommendation reasoning.


PHASE 1 - Persona-Item Compatibility Engine
We’ll compute compatibility using:

- dominant values	(ambience vs affordability).
- cuisine preferences	(likes Japanese food).
- communication identity	(expressive vs analytical).
- Nigerian identity	(soft-life vs practical).
- emotional style	(optimistic vs critical).

Then combine them into one interpretable compatibility score.

PHASE 1 ROADMAP

We’ll implement:

Step	Goal
1	- Build restaurant feature profiles
2	- Build cuisine preference matching
3	- Build value compatibility
4	- Build Nigerian identity matching
5	- Combine into unified score
6	- Generate reasoning explanations

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# STEP 1 — LOAD PERSONA ENGINE OUTPUT
persona_df = pd.read_csv(
    "../data/processed/persona_profiles.csv"
)

persona_df.head()

In [ ]:
# STEP 2 — LOAD YELP BUSINESS DATA
businesses = pd.read_json(
    "../data/raw/yelp_academic_dataset_business.json",
    lines=True
)

businesses.head()

In [ ]:
# STEP 3 — LOAD CURATED REVIEWS (IF SAVED)

curated_reviews = pd.read_csv(
    "../data/processed/curated_reviews.csv"
)

curated_reviews.head()

In [ ]:
# STEP 3 — CREATE BUSINESS FEATURE TABLE

business_profiles = businesses[
    [
        "business_id",
        "name",
        "categories",
        "stars",
        "review_count"
    ]
].copy()

business_profiles.head()

In [ ]:
business_profiles = businesses[
    [
        "business_id",
        "name",
        "categories",
        "stars",
        "review_count"
    ]
].copy()

In [ ]:
# STEP 4 — CLEAN CATEGORY TEXT

business_profiles["categories"] = (
    business_profiles["categories"]
    .fillna("")
)

In [ ]:
# STEP 5 - CREATE BUSINESS ATTRIBUTES

def extract_business_traits(category_text):

    text = category_text.lower()

    traits = {

        "ambience": 0,
        "luxury": 0,
        "convenience": 0,
        "social": 0,
        "casual": 0,
        "food_focus": 0,
        "shopping": 0,
        "time_efficiency": 0,
        "durability": 0
    }

    ambience_words = [
        "lounges", "cafe", "cafes", "rooftop", "wine", "cocktail", "ambience", "atmosphere", "decor", "vibes", 
        "music", "aesthetic", "cozy", "lighting", "cleanliness", "noise level", "comfortable seating", 
        "romantic", "family-friendly", "decoration choke", "overcrowded", "spacious", "intimate", "loud", 
        "quiet", "coffee", "tea", "desserts", "brunch", "bakery", "garden"
    
    ]

    luxury_words = [
        "fine dining", "steakhouse", "upscale", "premium", "luxury", "club", "sports bar", "golf course",
        "upscale", "fancy", "high-end", "exclusive", "luxurious", "opulent", 
        "lavish", "posh", "sophisticated", "elegant", "glamorous", "hotel", 
        "resort", "spa", "gourmet", "Michelin", "sommelier", "champagne", "caviar"
    ]

    convenience_words = [
        "fast", "quick", "parking", "location", "accessible", "easy",  "waiting time"
        "near me", "home delivery", "takeaway", "drive-thru", "curbside pickup", "self-service"
    ]

    social_words = [
        "bars", "nightlife", "music", "clubs", "friends", "family", "date", "group", "celebration", "birthday", "hangout"
        "social gathering", "romantic dinner", "family outing", "friend meetup", "special occasion"
        "anniversary", "reunion", "casual hangout", "work event", "holiday celebration"
        "new spot to try", "place to see and be seen", "vibe for socializing", "perfect for groups", "intimate setting",
        "sports bars", "karaoke", "beer", "pub"
    ]

    casual_words = [
        "fast food", "pizza", "burgers", "sandwiches", "food trucks", "takeout"
    ]

    food_words = [
        "restaurants", "seafood", "sushi", "bbq", "ramen", "mexican", "italian", "thai", 
        "korean", "indian", "chinese", "vegetarian", "vegan", "gluten-free", "desserts", "brunch", "bakery"
    ]

    shopping_words = [
        "groceries", "home", "kitchen", "electronics"
    ]

    time_words = [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time", "arrived late", 
        "arrived early", "on schedule", "behind schedule", "ahead of schedule", "fast"
    ]

    durability_words = [
        "original", "fake", "counterfeit", "rugged", "last long", "strong",
        "weak", "fragile", "repair", "spare parts", "generator", "battery life",
        "heat up", "spoilt", "still working", "tested and trusted", "tokunbo", "new", "used"
    ]

    for word in ambience_words:
        if word in text:
            traits["ambience"] += 1

    for word in luxury_words:
        if word in text:
            traits["luxury"] += 1

    for word in social_words:
        if word in text:
            traits["social"] += 1

    for word in casual_words:
        if word in text:
            traits["casual"] += 1

    for word in food_words:
        if word in text:
            traits["food_focus"] += 1

    for word in shopping_words:
        if word in text:
            traits["shopping"] += 1
    
    for word in time_words:
        if word in text:
            traits["time_efficiency"] += 1

    for word in convenience_words:
        if word in text:
            traits["convenience"] += 1

    for word in durability_words:
        if word in text:
            traits["durability"] += 1

    return traits

In [ ]:
# STEP 6 — APPLY BUSINESS TRAIT EXTRACTION

business_profiles["traits"] = (
    business_profiles["categories"]
    .fillna("")
    .apply(extract_business_traits)
)

In [ ]:
# STEP 7 — EXPAND BUSINESS TRAITS

business_traits_df = pd.json_normalize(
    business_profiles["traits"]
)

business_traits_df.head()

In [ ]:
# STEP 8 — MERGE TRAITS INTO BUSINESS TABLE
# Transforming raw business metadata into interpretable environmental signals.

business_profiles = pd.concat(
    [business_profiles, business_traits_df],
    axis=1
)

business_profiles.head(10)

In [ ]:
# STEP 9 — CREATE VALUE COMPATIBILITY FUNCTION
# We now compare user values VS restaurant traits.

def value_compatibility(persona_row, business_row):

    score = 0

    dominant_value = persona_row["dominant_value"]

    if dominant_value == "ambience":
        score += business_row["ambience"]

    elif dominant_value == "social_experience":
        score += business_row["social"]

    elif dominant_value == "food_quality":
        score += business_row["food_focus"]

    elif dominant_value == "affordability":
        score += business_row["casual"]

    elif dominant_value == "time":
            score += business_row["time_efficiency"]

    elif dominant_value == "convenience":
            score += business_row["convenience"]

    elif dominant_value == "durability":
            score += business_row["durability"]

    elif dominant_value == "shopping":
            score += business_row["shopping"]

    return score

In [ ]:
# STEP 10 — CREATE NIGERIAN COMPATIBILITY FUNCTION
# cultural alignment.
# We can create a function that assesses how well a business aligns with the specific cultural identities we've identified in our Nigerian personas. 
# This function will take into account the unique traits and preferences associated with each Nigerian identity and compare them against the features of the businesses.


def nigerian_compatibility(persona_row, business_row):

    identity = persona_row["nigerian_identity"]

    score = 0

    if identity == "soft_life_explorer":
        score += (
            business_row["luxury"]
            + business_row["ambience"]
        )

    elif identity == "social_enjoyment":
        score += business_row["social"]

    elif identity == "practical_survivor":
        score += business_row["casual"]

    elif identity == "time_efficiency":
        score += business_row["time_efficiency"]

    elif identity == "pidgin_staples":
        score += business_row["casual"] + business_row["social"]

        
    return score

In [ ]:
# STEP 11 — CREATE COMPATIBILITY ENGINE

def compute_compatibility(
    persona_row,
    business_row
):

    value_score = value_compatibility(
        persona_row,
        business_row
    )

    nigerian_score = nigerian_compatibility(
        persona_row,
        business_row
    )

    final_score = (
        value_score * 0.7
        + nigerian_score * 0.3
    )

    return {
        "value_score": value_score,
        "nigerian_score": nigerian_score,
        "final_score": final_score
    }

In [ ]:
# Remove duplicate columns

business_profiles = business_profiles.loc[
    :,
    ~business_profiles.columns.duplicated()
]

In [ ]:
# STEP 12 — TEST COMPATIBILITY ENGINE

# Now we test with one persona + one restaurant

sample_persona = persona_df.iloc[7]

sample_business = business_profiles.iloc[81]

compute_compatibility(
    sample_persona,
    sample_business
)

In [ ]:
# 12a: Get persona details
# This helps you retrieve persona information to test for compatibility with a business

persona_df[
    [
        "user_id",
        "archetype",
        "dominant_value",
        "nigerian_identity",
        "communication_style"
    ]
].head(40)

In [ ]:
# Get specific persona user ID

target_user_id = "1kdfj_PaRk8i870ghdIvXg"

In [ ]:
target_persona = persona_df[
    persona_df["user_id"] == target_user_id
]

target_persona.T

In [ ]:
# Get selected persona user review to get insight into their preference
user_reviews = curated_reviews[
    curated_reviews["user_id"] == target_user_id
]

user_reviews[
    [
        "stars",
        "categories",
        "text"
    ]
].head(5)

In [ ]:
#Business Information

candidate_businesses = business_profiles[
    business_profiles["categories"]
    .str.contains(
        "Restaurants|Spanish|Italian|Steakhouses|Chicken Wings",
        case=False,
        na=False
    )
]

candidate_businesses[
    [
        "business_id",
        "name",
        "categories",
        "ambience",
        "luxury",
        "social"
    ]
].head(20)

In [ ]:
business_profiles.iloc[35]

In [ ]:
#Select Business ID
target_business = candidate_businesses.iloc[93]

In [ ]:
compatibility = compute_compatibility(
    target_persona.iloc[0],
    target_business
)

compatibility

In [ ]:
# STEP 13 — GENERATE HUMAN-READABLE REASONING

def explain_compatibility(
    persona_row,
    business_row,
    compatibility_result
):

    reasons = []

    if compatibility_result["value_score"] > 0:

        reasons.append(
            f"Matches the user's value for "
            f"{persona_row['dominant_value']}"
        )

    if compatibility_result["nigerian_score"] > 0:

        reasons.append(
            f"Aligns with the user's "
            f"Nigerian identity "
            f"({persona_row['nigerian_identity']})"
        )

    if len(reasons) == 0:

        reasons.append(
            "Limited behavioral alignment detected"
        )

    return reasons

In [ ]:
# STEP 14 - TEST EXPLANATION ENGINE
compatibility = compute_compatibility(
    sample_persona,
    sample_business
)

explain_compatibility(
    sample_persona,
    sample_business,
    compatibility
)

PHASE 1 — Persona-Item Compatibility Engine is now Complete

The agent can now evaluate behavioral alignment, reason about compatibility, explain recommendations and model culturally-aware preference matching

PHASE 2: CONTEXT TAXONOMY

In [ ]:
# STEP 15 — CREATE CONTEXT TAXONOMY
# modeling situational psychology.

context_taxonomy = {

    "weekday_quick_meal": {

        "prefers": [
            "casual",
            "affordability"
        ],

        "avoids": [
            "luxury"
        ],

        "mood": "busy",

        "group_size": "just_me",

        "economic_context": "budget_conscious"
    },

    "social_night_out": {

        "prefers": [
            "social",
            "ambience"
        ],

        "avoids": [],

        "mood": "energetic",

        "group_size": "small_group",

        "economic_context": "flexible_budget"
    },

    "celebration": {

        "prefers": [
            "luxury",
            "ambience"
        ],

        "avoids": [
            "casual"
        ],

        "mood": "celebratory",

        "group_size": "group",

        "economic_context": "splurging"
    },

    "comfort_food_mood": {

        "prefers": [
            "casual",
            "food_focus",
            "authentic_local"
        ],

        "avoids": [],

        "mood": "tired",

        "group_size": "just_me",

        "economic_context": "normal"
    },

    "work_cafe_session": {

        "prefers": [
            "ambience",
            "quiet"
        ],

        "avoids": [
            "social"
        ],

        "mood": "focused",

        "group_size": "just_me",

        "economic_context": "moderate"
    }
}

In [ ]:
# CONTEXT DIMENSIONS
occasion_types = [

    "casual", "business_meeting", "formal", "celebration", "quick_bite", "takeaway", "delivery",
    "brunch", "after_work", "late_night"
]

group_sizes = [

    "just_me", "couple", "small_group", "large_group",
    "crowd"
]

moods = [

    "happy", "celebratory", "relaxed", "stressed", "tired", "hungry",
    "excited", "romantic", "chilled", "focused"
]

lagos_locations = [

    "Victoria_Island", "Lekki", "Ikoyi", "Yaba", "Surulere",
    "Ikeja", "Ajah", "Mainland", "Island"
]

economic_contexts = [

    "salary_day", "sapa_period", "splurging", "budget_conscious", "treat_yourself"
]

In [ ]:
#STEP 16 — CREATE CONTEXT COMPATIBILITY FUNCTION
# Now we score business fit for current context.

def context_compatibility(
    business_row,
    context_name
):

    context = context_taxonomy[context_name]

    score = 0

    for preference in context["prefers"]:

        if preference in business_row:
            score += business_row[preference]

    for avoidance in context["avoids"]:

        if avoidance in business_row:
            score -= business_row[avoidance]

    return score

In [ ]:
business_profiles.iloc[153]

In [ ]:
# STEP 17 — TEST CONTEXTUAL DIFFERENCES

sample_business = business_profiles.iloc[153]

for context_name in context_taxonomy.keys():

    score = context_compatibility(
        sample_business,
        context_name
    )

    print(context_name, "→", score)

The same restaurant now scores differently depending on context.

This is dynamic reasoning.

In [ ]:
# STEP 18 — CREATE CONTEXT-AWARE COMPATIBILITY ENGINE
# Now combine persona compatibility and situational compatibility

def contextualized_compatibility(
    persona_row,
    business_row,
    context_name
):

    base_scores = compute_compatibility(
        persona_row,
        business_row
    )

    context_score = context_compatibility(
        business_row,
        context_name
    )

    final_score = (
        base_scores["final_score"] * 0.7
        + context_score * 0.3
    )

    return {

        "base_score":
            base_scores["final_score"],

        "context_score":
            context_score,

        "final_score":
            final_score
    }

In [ ]:
# STEP 19 — TEST CONTEXTUALIZED REASONING
# Now we see how the same business can score differently for the same persona under different contexts.

sample_persona = persona_df.iloc[7]

sample_business = business_profiles.iloc[9]

contextualized_compatibility(
    sample_persona,
    sample_business,
    "celebration" # can be changed to other contexts like "social_night_out", "celebration", "work_cafe_session"
)

In [ ]:
contextualized_compatibility(
    sample_persona,
    sample_business,
    "weekday_quick_meal"
)

The SAME user, in the same restaurant now produces different compatibility scores depending on situation.

This is akin to human-like reasoning.

In [ ]:
# STEP 19 — GENERATE CONTEXTUAL EXPLANATIONS
# Now we can explain the reasoning behind the context-aware compatibility scores in a human-readable way.

def explain_contextual_reasoning(
    persona_row,
    business_row,
    context_name,
    result
):

    explanation = []

    explanation.append(
        f"Context: {context_name}"
    )

    if result["context_score"] > 0:

        explanation.append(
            "This venue aligns well with the current situation."
        )

    else:

        explanation.append(
            "This venue may not strongly fit the current situation."
        )

    explanation.extend(

        explain_compatibility(
            persona_row,
            business_row,
            compute_compatibility(
                persona_row,
                business_row
            )
        )
    )

    return explanation

In [ ]:
# STEP 20 — TEST CONTEXTUAL EXPLANATION

result = contextualized_compatibility(
    sample_persona,
    sample_business,
    "celebration"
)

explain_contextual_reasoning(
    sample_persona,
    sample_business,
    "celebration",
    result
)

PHASE 2 — Contextual + Cultural Reasoning is now complete

The agent can now: reason contextually
- understand Nigerian cultural signals
- adapt recommendations culturally
- model situational behavior
- align personas with local behavioral styles

PHASE 3 — RATING PREDICTION ENGINE
Before humans write reviews they already internally decide, “How good or bad was this experience?”

That becomes the star rating. The written review is usually a justification of that score.

So our architecture becomes: 
persona + context + restaurant compatibility + emotional state = predicted rating = generated review

BUILD:
heuristic rating prediction, 
compatibility-driven scoring, 
personality-adjusted ratings, 
emotional drift effects, 
contextual rating modifiers.

In [ ]:
# STEP 21 — CREATE RATING PREDICTION FUNCTION
# Finally, we can create a function that translates the compatibility scores into a predicted star rating (1-5) that the user might give to the business.
def base_rating_prediction(
    compatibility_score
):

    if compatibility_score >= 3:
        return 5

    elif compatibility_score >= 2:
        return 4

    elif compatibility_score >= 1:
        return 3

    elif compatibility_score >= 0:
        return 2

    else:
        return 1
    
# This creates : compatibility → satisfaction mapping.

In [ ]:
# STEP 22 — ADD PERSONALITY RATING BIAS
# We can also add a bias factor based on the user's communication style or personality traits that we've identified in the persona profiles. 
# For example, some personas might be more likely to give higher ratings due to their optimistic nature, while others might be more critical. 
# This bias can be a simple adjustment to the predicted rating based on the persona's archetype or communication style.
# Harsh Critic → harsher ratings
# Warm Optimist → forgiving ratings

personality_rating_bias = {

    "Warm Optimist": 0.5,

    "Reactive Reviewer": 0,

    "Harsh Critic": -1,

    "Emotional Storyteller": 0.3,

    "Deep Experience Analyst": -0.3
}

In [ ]:
# Step 23 — CREATE PERSONALITY-ADJUSTED RATING PREDICTION FUNCTION
# This function takes the base compatibility score, converts it to a rating, and then adjusts it based on the persona's archetype bias.

def personality_adjusted_rating(
    persona_row,
    compatibility_score
):

    base_rating = base_rating_prediction(
        compatibility_score
    )

    bias = personality_rating_bias[
        persona_row["archetype"]
    ]

    adjusted = base_rating + bias

    adjusted = round(adjusted)

    adjusted = max(1, min(5, adjusted))

    return adjusted

In [ ]:
# STEP 24 — TEST PERSONALITY-ADJUSTED RATING PREDICTION 
# Now we can see how the same compatibility score might translate into different predicted ratings for different personas based on their archetype biases.

compatibility_score = 2

for archetype in personality_rating_bias.keys():

    persona_sample = persona_df[
        persona_df["archetype"] == archetype
    ].iloc[0]

    prediction = personality_adjusted_rating(
        persona_sample,
        compatibility_score
    )

    print(archetype, "→", prediction)

The SAME experience produces different ratings depending on personality. This is behavioral realism.

In [ ]:
# STEP 25 — ADD CONTEXT RATING BIAS
# We can also add a bias based on the current context, as some contexts might lead to 
# higher or lower ratings on average. For example, people might be more forgiving in a celebratory context, 
# but more critical in a stressful or work-related context.

context_rating_bias = {

    "celebration": 0.5,

    "social_night_out": 0.3,

    "comfort_food_mood": 0.2,

    "weekday_quick_meal": -0.2,

    "work_cafe_session": -0.1
}

In [ ]:
# Step 26 — CREATE CONTEXT-AWARE RATING PREDICTION FUNCTION
# This function combines the base compatibility score, the personality bias, and the context bias to produce a final predicted rating.

def contextual_rating_prediction(
    persona_row,
    compatibility_result,
    context_name
):

    compatibility_score = (
        compatibility_result["final_score"]
    )

    personality_rating = (
        personality_adjusted_rating(
            persona_row,
            compatibility_score
        )
    )

    context_bias = (
        context_rating_bias[context_name]
    )

    final_rating = (
        personality_rating + context_bias
    )

    final_rating = round(final_rating)

    final_rating = max(1, min(5, final_rating))

    return final_rating

In [ ]:
# STEP 27 — TEST CONTEXTUAL RATING PREDICTION
# Now we can see how the same business might receive different predicted ratings from the same persona under different contexts due to the context bias.

sample_persona = persona_df.iloc[10]

sample_business = business_profiles.iloc[70]

compatibility_result = (
    contextualized_compatibility(
        sample_persona,
        sample_business,
        "celebration"
    )
)

contextual_rating_prediction(
    sample_persona,
    compatibility_result,
    "celebration"
)

In [ ]:
contextual_rating_prediction(
    sample_persona,
    compatibility_result,
    "weekday_quick_meal"
)

In [ ]:
# STEP 28 — EXPLAIN RATING PREDICTION
# Finally, we can create an explanation function that breaks down the reasoning behind the 
# predicted rating in a human-readable way, incorporating the persona's archetype, the context, and the compatibility scores.

def explain_rating_prediction(
    persona_row,
    context_name,
    predicted_rating
):

    explanation = []

    explanation.append(
        f"Predicted rating: {predicted_rating} stars"
    )

    explanation.append(
        f"User archetype: "
        f"{persona_row['archetype']}"
    )

    explanation.append(
        f"Context: {context_name}"
    )

    if predicted_rating >= 4:

        explanation.append(
            "Strong behavioral alignment detected."
        )

    elif predicted_rating == 3:

        explanation.append(
            "Moderate alignment with mixed signals."
        )

    else:

        explanation.append(
            "Weak alignment or dissatisfaction likely."
        )

    return explanation

In [ ]:
# STEP 28 — TEST EXPLANATION OF RATING PREDICTION
# Now we can see how the explanation function breaks down the reasoning behind the predicted rating in a human-readable way.

predicted_rating = (
    contextual_rating_prediction(
        sample_persona,
        compatibility_result,
        "celebration"
    )
)

explain_rating_prediction(
    sample_persona,
    "celebration",
    predicted_rating
)

PHASE 3 — Rating Prediction Engine is now complete

The system can now:

- simulate satisfaction
- predict ratings behaviorally
- adjust scores contextually
- reflect personality biases
- explain predicted decisions

PHASE 4 — REVIEW PLANNING ENGINE
Humans experience an internal planning process to writinf reviews.
Before writing people decide:

- what mattered most
- what emotion dominated
- whether to rant or praise
- whether to be brief or detailed
- whether to sound analytical or expressive

So the BUILD for this phase will comprise:

- review tone mapping
- emotional intensity
- verbosity
- narrative structure
- criticism style
- praise emphasis
- Nigerian conversational flavor

In [ ]:
# STEP 29 — CREATE TONE MAPPING
# this maps the predicted star ratings to a tone that could be used in a review generation 
# system to create more personalized and context-aware reviews based on the predicted sentiment 
# of the user towards the business.
tone_mapping = {

    5: "enthusiastic",

    4: "positive",

    3: "balanced",

    2: "disappointed",

    1: "frustrated"
}

In [ ]:
# STEP 30 — TEST TONE MAPPING. CREATE EMOTIONAL INTENSITY MAPPING
# for rating in range(1, 6):

    # tone = tone_mapping[rating]

    # print(f"{rating} stars → {tone}")

archetype_intensity = {

    "Warm Optimist": "medium",

    "Reactive Reviewer": "high",

    "Harsh Critic": "high",

    "Emotional Storyteller": "very_high",

    "Deep Experience Analyst": "low"
}

In [ ]:
# STEP 31 — CREATE VERBOSITY MAPPING
# This maps the archetypes to a verbosity level that could be used in a review generation 
# system to create more personalized and archetype-consistent reviews based on the user's communication style.

verbosity_mapping = {

    "Warm Optimist": "medium",

    "Reactive Reviewer": "short",

    "Harsh Critic": "medium",

    "Emotional Storyteller": "long",

    "Deep Experience Analyst": "very_long"
}

In [ ]:
# STEP 32 — CREATE CRITICISM STYLE MAPPING
# This maps the archetypes to a criticism style that could be used in a review generation system 
# to create more personalized and archetype-consistent reviews based on the user's communication style.

criticism_style_mapping = {

    "Warm Optimist":
        "forgiving",

    "Reactive Reviewer":
        "emotionally_reactive",

    "Harsh Critic":
        "direct",

    "Emotional Storyteller":
        "dramatic",

    "Deep Experience Analyst":
        "analytical"
}

In [ ]:
# STEP 33 — CREATE NIGERIAN IDENTITY STYLE MAPPING
# This maps the Nigerian identities to a style that could be used in a review generation system 
# to create more personalized and culturally resonant reviews based on the user's specific Nigerian identity.

nigerian_flavor_mapping = {

    "soft_life":
        "luxury_lagos_style",

    "social_vibes":
        "expressive_lagos_style",

    "value_sensitive":
        "practical_nigerian_style",

    "hustle_minded":
        "hustle_nigerian_style",

    "sarcastic_dramatic":
        "sarcastic_nigerian_style",

    "neutral":
        "generic_nigerian_style"
}

In [ ]:
nigerian_persona_map = {

    "Warm Optimist":
        "social_vibes",

    "Reactive Reviewer":
        "value_sensitive",

    "Harsh Critic":
        "value_sensitive",

    "Emotional Storyteller":
        "soft_life",

    "Deep Experience Analyst":
        "soft_life"
}

persona_df["nigerian_style"] = (
    persona_df["archetype"]
    .map(nigerian_persona_map)
)

In [ ]:
# STEP 34 — BUILD REVIEW PLAN
# Now we can create a function that takes the persona information, the predicted rating, 
# and other relevant details to build a review generation plan that includes the tone, 
# emotional intensity, verbosity, criticism style, and Nigerian flavor that should be used 
# when generating a review for this user-business interaction.

def build_review_plan(
    persona_row,
    predicted_rating
):

    archetype = persona_row["archetype"]

    nigerian_style = (
        persona_row["nigerian_style"]
    )

    review_plan = {

        "tone":
            tone_mapping[predicted_rating],

        "emotional_intensity":
            archetype_intensity[archetype],

        "verbosity":
            verbosity_mapping[archetype],

        "criticism_style":
            criticism_style_mapping[archetype],

        "nigerian_flavor":
            nigerian_flavor_mapping[
                nigerian_style
            ],

        "predicted_rating":
            predicted_rating
    }

    return review_plan

In [ ]:
# STEP 35 — TEST REVIEW PLAN BUILDING
# Now we can see how the review plan is built based on the persona's archetype and the 
# predicted rating, which will guide the review generation system in creating a personalized 
# and context-aware review.

sample_persona = persona_df.iloc[4]

predicted_rating = 4

build_review_plan (
    sample_persona,
    predicted_rating
)

In [ ]:
# STEP 36 — CREATE REVIEW OPENING TEMPLATES
# Finally, we can create a set of opening sentence templates for reviews that correspond to different tones and archetypes. 
# These templates can be used by the review generation system to create more personalized and archetype-consistent reviews based on the user's predicted sentiment and communication style.

opening_styles = {

    "enthusiastic": [
        "Absolutely loved this place.",
        "This spot exceeded expectations.",
        "One of the best experiences I've had."
        "Really enjoyed my visit here.",
        "Solid experience overall.",
        "Had a good time here."
    ],

    "excited_positive": [
        "I'm still smiling as I write this.",
        "Chai! This place surprised me in the best way.",
        "Where do I even start? Absolute gem!",
        "See glass! This one na correct spot.",
        "If you no try this place, you're missing o.",
        "I dey feel good just remembering this experience.",
        "Omo, this place is a vibe from start to finish.",
        "Finally, somewhere that gets it right."
        "I am giving 5 stars because there is no room for 6"
    ],

    "positive": [
        "Really enjoyed my visit here.",
        "Solid experience overall.",
        "Had a good time here."
    ],
    
    "satisfied_content": [
        "No wahala experience from start to end.",
        "I genuinely enjoyed my time here.",
        "Solid 4 stars – no complaints, just small observations.",
        "It does what it says on the tin. Reliable.",
        "You know that feeling when everything just works? That was here.",
        "Nothing too fancy, but very solid.",
        "I would happily come back anytime."
    ],
    
    "balanced": [
        "Mixed feelings about this place.",
        "Some things worked, others didn't.",
        "Decent experience overall."
    ],

    "disappointed_negative": [
        "I regret stepping foot in this place.",
        "Nawa o. Where do I even start?",
        "This one hurt my pocket and my spirit.",
        "See disappointment. Chai!",
        "I asked myself 'why did I come here?' the whole time.",
        "Never again. I mean it.",
        "The only thing worse than the food was the service.",
        "Story for the gods – and not the good kind."
        "Expected much better honestly.",
        "Left somewhat disappointed.",
        "The experience was underwhelming."
    ],
    
    "angry_frustrated": [
        "I am fuming as I type this.",
        "This is the worst customer service I have ever received.",
        "See wahala! They have nerve o.",
        "I want my money back and my time back.",
        "Who send me? Abeg, avoid this place.",
        "E shock me that this place is still in business.",
        "I shouted at them – and I never shout."
        "This experience was genuinely frustrating.",
        "Would not return after this visit.",
        "Very disappointing experience."
    ],
    
    "sarcastic_playful": [
        "Oh wow. Just wow. (Sarcasm fully intended).",
        "Congratulations to them for the audacity.",
        "If zero stars was possible, I would give it.",
        "The only good thing was leaving.",
        "I’m not sure if the cook was angry at me personally.",
        "This place is… an experience. Not a good one.",
        "They tried. They failed. But they tried."
    ],
    
    "surprised_mixed": [
        "I didn't expect much, but wow – I was wrong.",
        "Honestly, I'm confused about how to rate this.",
        "It had good parts and bad parts. Let me explain.",
        "First half was terrible. Second half was amazing.",
        "My feelings are still conflicted.",
        "I wanted to love it, but..."
    ],
    
    "neutral_factual": [
        "Just the facts: here is my experience.",
        "No hype, no hate – just an honest review.",
        "I'll keep it short and straightforward.",
        "Let me break it down without sugarcoating.",
        "Here is what worked and what didn't."
    ],
    
    "family_oriented": [
        "Took the whole family there – here's how it went.",
        "My kids loved it, but my wallet didn't.",
        "I went with my people and we had a good time overall.",
        "This is a good spot for group outings.",
        "Even my uncle who complains about everything liked it."
    ],
    
    "romantic_date": [
        "Perfect date night spot – trust me.",
        "Took my babe here and the vibes were right.",
        "If you want to impress your partner, bring them here.",
        "Romantic, quiet, and classy. 10/10 for couples.",
        "The ambience alone is worth the visit."
    ],
    
    "hustle_budget": [
        "On a budget? This place won't kill your wallet.",
        "I went during sapa period and still managed to enjoy.",
        "Value for money? Yes. Luxury? No. Fair trade.",
        "Cheap and cheerful – nothing more, nothing less.",
        "My money didn't cry after leaving here."
    ],
    
    "hangry_urgent": [
        "I was starving when I arrived, so maybe I'm biased.",
        "They fed a hungry person – that counts for something.",
        "I almost lost my cool waiting, but the food saved them.",
        "If you're very hungry, this place will do.",
        "Fast service for a starving customer – thank you."
    ],
    
    "after_work_tired": [
        "Came here after a long day – just wanted to relax.",
        "I was too tired to complain, but honestly it was fine.",
        "Decent place to unwind after work.",
        "Low energy, high patience – and they delivered."
    ]
}

In [ ]:
# STEP 37 — CREATE FOCUS AREA GENERATOR 
# Now we can see how the review opening templates correspond to different tones, focus, and 
# archetypes, and we can create a function that generates a focus area for the review 
# based on the user's dominant value, which will guide the content of the review to 
# align with what matters most to the user.

def determine_review_focus(
    persona_row
):

    dominant_value = (
        persona_row["dominant_value"]
    )

    if dominant_value == "ambience":

        return [
            "atmosphere", "aesthetics", "vibes", "lighting", "music", 
            "decor", "seating comfort", "view"
        ]

    elif dominant_value == "food_quality":

        return [
            "taste", "food quality", "portion sizes"
        ]

    elif dominant_value == "social_experience":

        return [
            "social energy", "music", "crowd", "energy", "DJ quality", "crowd liveliness", "drink selection",
            "dance floor", "after-hours", "security", "nightlife", "party atmosphere"
        ]

    elif dominant_value == "affordability":

        return [
            "pricing", "value", "price", "value for money", "cost", "budget-friendliness",
            "portion size vs price", "hidden charges", "discounts"
        ]

    elif dominant_value == "durability":

        return [
            "build quality", "longevity", "ruggedness", "original vs fake",
            "materials", "resistance to wear", "repairability"
        ]

    elif dominant_value == "service_quality":

        return [
            "staff attitude", "speed of service", "politeness", "attentiveness",
            "problem resolution", "wait time", "follow-up"
        ]

    elif dominant_value == "social_proof":

        return [
            "crowd popularity", "friend recommendations", "family approval",
            "neighbour's experience", "busyness", "word-of-mouth reputation"
        ]

    elif dominant_value == "time_efficiency":

        return [
            "waiting time", "delivery speed", "queue length", "preparation time",
            "punctuality", "response time"
        ]

    elif dominant_value == "luxury":

        return [
            "premium experience", "exclusivity", "high-end finishes", "brand prestige",
            "attentive VIP treatment", "aesthetics", "expensive but worth"
        ]
    
    elif dominant_value == "romance":

        return [
            "intimacy", "candlelight", "quiet corners", "couple seating",
            "date night suitability", "rosy atmosphere"
        ]

    elif dominant_value == "family_friendliness":

        return [
            "kid-friendly", "space for groups", "family seating", "child menu",
            "play area", "stroller access"
        ]

    elif dominant_value == "authenticity":

        return [
            "traditional recipes", "original preparation", "cultural accuracy",
            "local ingredients", "home-style cooking", "no shortcuts"
        ]


    return ["overall experience"]

In [ ]:
determine_review_focus(
    sample_persona
)

In [ ]:
# STEP 38 — GENERATE REVIEW BLUEPRINT
# Finally, we can create a function that takes all the previous components — 
# the persona information, the predicted rating, the review plan, and the focus areas — 
# to generate a comprehensive blueprint for how a review should be generated for this 
# user-business interaction, which can then be used by a review generation system to 
# create a personalized, context-aware, and archetype-consistent review.

def generate_review_blueprint(
    persona_row,
    predicted_rating
):

    plan = build_review_plan(
        persona_row,
        predicted_rating
    )

    focus_areas = determine_review_focus(
        persona_row
    )

    blueprint = {

        "review_plan": plan,

        "focus_areas": focus_areas,

        "opening_examples":
            opening_styles[
                plan["tone"]
            ]
    }

    return blueprint

In [ ]:
generate_review_blueprint(
    sample_persona,
    predicted_rating
)

PHASE 4 — Review Planning Engine is now complete.
The system now has:

- Human personas	
- Value systems	
- Emotional drift	
- Nigerian contextualization
- Compatibility reasoning	
- Contextual reasoning	
- Rating prediction	
- Review planning

In [ ]:
# PHASE 5 — ACTUAL REVIEW GENERATION